In [36]:
from google.colab import files

print('Upload your Hindi audio file (.wav or .mp3)')
uploaded = files.upload()
audio_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {audio_filename}')

# GROUND TRUTH: type what you actually said (for rough WER calculation)
# Change this to match your recording!
GROUND_TRUTH = 'वाराणसी में सबसे प्रसिद्ध घाट कौन सा है'

Upload your Hindi audio file (.wav or .mp3)


Saving Audio10.mp3 to Audio10 (2).mp3
✅ Uploaded: Audio10 (2).mp3


In [37]:
!pip install -q openai-whisper sentence-transformers faiss-cpu gtts
print('✅ All packages installed: whisper, sentence-transformers, faiss-cpu, gtts')

✅ All packages installed: whisper, sentence-transformers, faiss-cpu, gtts


In [38]:
import whisper
import time

# ---- SMALL MODEL ----
print('Loading whisper-small...')
model_small = whisper.load_model('small')
t0 = time.time()
result_small = model_small.transcribe(
    audio_filename,
    language='hi',
    task='transcribe',
    fp16=False,
    temperature=0.0
)
t_small = time.time() - t0

# ---- MEDIUM MODEL ----
print('Loading whisper-medium...')
model_medium = whisper.load_model('medium')
t0 = time.time()
result_medium = model_medium.transcribe(
    audio_filename,
    language='hi',
    task='transcribe',
    fp16=False,
    temperature=0.0
)
t_medium = time.time() - t0

print('\n' + '='*60)
print('📊 ASR COMPARISON')
print('='*60)
print(f'Ground truth : {GROUND_TRUTH}')
print(f'Small  ({t_small:.1f}s) : {result_small["text"].strip()}')
print(f'Medium ({t_medium:.1f}s) : {result_medium["text"].strip()}')
print('='*60)

# Use medium model for rest of pipeline
transcribed_query = result_medium['text'].strip()
print(f'\n✅ Using medium model output: "{transcribed_query}"')

Loading whisper-small...
Loading whisper-medium...

📊 ASR COMPARISON
Ground truth : वाराणसी में सबसे प्रसिद्ध घाट कौन सा है
Small  (2.2s) : वार्जनासी में सब से प्रसिद गार्ट कोंचा है
Medium (2.1s) : वारनसी में सबसे प्रसिड घात कौनसा है?

✅ Using medium model output: "वारनसी में सबसे प्रसिड घात कौनसा है?"


In [39]:
def word_error_rate(reference, hypothesis):
    ref_words = reference.strip().split()
    hyp_words = hypothesis.strip().split()

    d = [[0] * (len(hyp_words) + 1) for _ in range(len(ref_words) + 1)]
    for i in range(len(ref_words) + 1):
        d[i][0] = i
    for j in range(len(hyp_words) + 1):
        d[0][j] = j
    for i in range(1, len(ref_words) + 1):
        for j in range(1, len(hyp_words) + 1):
            cost = 0 if ref_words[i-1] == hyp_words[j-1] else 1
            d[i][j] = min(d[i-1][j] + 1,
                          d[i][j-1] + 1,
                          d[i-1][j-1] + cost)

    return round(d[len(ref_words)][len(hyp_words)] / len(ref_words) * 100, 1)

wer_small  = word_error_rate(GROUND_TRUTH, result_small['text'])
wer_medium = word_error_rate(GROUND_TRUTH, result_medium['text'])

print('\n' + '='*50)
print('📏 WORD ERROR RATE')
print('='*50)
print(f'Ground truth   : {GROUND_TRUTH}')
print(f'Small  output  : {result_small["text"].strip()}')
print(f'Medium output  : {result_medium["text"].strip()}')
print(f'whisper-small  WER : {wer_small}%')
print(f'whisper-medium WER : {wer_medium}%')
print('='*50)


📏 WORD ERROR RATE
Ground truth   : वाराणसी में सबसे प्रसिद्ध घाट कौन सा है
Small  output  : वार्जनासी में सब से प्रसिद गार्ट कोंचा है
Medium output  : वारनसी में सबसे प्रसिड घात कौनसा है?
whisper-small  WER : 75.0%
whisper-medium WER : 75.0%


In [45]:
# Expand this freely — the more sentences, the better the retrieval
contexts = [
    # Varanasi / Ghats
    'वाराणसी में सबसे प्रसिद्ध घाट दशाश्वमेध घाट है, जहाँ प्रतिदिन गंगा आरती होती है।',
        'वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।',
    'मणिकर्णिका घाट वाराणसी का सबसे पवित्र श्मशान घाट है।',
    'वाराणसी में 84 घाट गंगा नदी के किनारे स्थित हैं।',
    'वाराणसी को काशी और बनारस के नाम से भी जाना जाता है — यह भारत का सबसे पुराना शहर है।',
    'काशी विश्वनाथ मंदिर वाराणसी में स्थित है और भगवान शिव को समर्पित है।',
    # Ramayana / Hanuman
    'हनुमान जी भगवान राम के परम भक्त और सेवक थे।',
    'हनुमान जी ने लंका में जाकर माता सीता की खोज की और उन्हें राम जी का संदेश दिया।',
    'हनुमान जी ने संजीवनी बूटी लाकर लक्ष्मण जी की जान बचाई थी।',
    'रावण ने माता सीता का अपहरण किया और उन्हें लंका ले गया।',
    'भगवान राम ने वानर सेना की मदद से रावण का वध किया।',
    # Ganga / Nature
    'गंगा नदी हिमालय के गंगोत्री ग्लेशियर से निकलती है।',
    'गंगा नदी को भारत की सबसे पवित्र नदी माना जाता है।',
    # Weather / General
    'वाराणसी में गर्मियों में तापमान 45 डिग्री तक पहुँच सकता है।',
    'मानसून जून से सितंबर तक उत्तर प्रदेश में रहता है।',
    # BHU
    'बनारस हिंदू विश्वविद्यालय (BHU) वाराणसी में स्थित है और एशिया के सबसे बड़े विश्वविद्यालयों में से एक है।',
]

print(f'✅ Knowledge base: {len(contexts)} sentences loaded')

✅ Knowledge base: 16 sentences loaded


In [50]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print('Loading sentence embedder (all-MiniLM-L6-v2)...')
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print('Encoding knowledge base...')
t0 = time.time()
embeddings = embedder.encode(contexts, show_progress_bar=True)
embeddings = np.array(embeddings, dtype='float32')
t_index = time.time() - t0

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f'✅ FAISS index built in {t_index:.2f}s | {index.ntotal} vectors | dim={dim}')

Loading sentence embedder (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding knowledge base...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built in 0.07s | 16 vectors | dim=384


In [51]:
def retrieve(query, k=2):
    """Retrieve top-k most relevant sentences for a Hindi query."""
    t0 = time.time()
    q_emb = embedder.encode([query], show_progress_bar=False)
    q_emb = np.array(q_emb, dtype='float32')
    distances, indices = index.search(q_emb, k=k)
    latency_ms = (time.time() - t0) * 1000

    results = [(contexts[i], distances[0][rank]) for rank, i in enumerate(indices[0])]
    return results, latency_ms

# Quick sanity check
test_results, lat = retrieve('हनुमान जी ने क्या किया?', k=2)
print(f'Sanity check retrieval ({lat:.1f}ms):')
for ctx, dist in test_results:
    print(f'  dist={dist:.3f} | {ctx}')

Sanity check retrieval (13.1ms):
  dist=10.529 | हनुमान जी ने संजीवनी बूटी लाकर लक्ष्मण जी की जान बचाई थी।
  dist=11.000 | हनुमान जी भगवान राम के परम भक्त और सेवक थे।


In [52]:
from gtts import gTTS
from IPython.display import Audio, display

def speak_response(response_text, filename='response.mp3'):
    """Convert Hindi text to speech and play it in Colab."""
    tts = gTTS(text=response_text, lang='hi', slow=False)
    tts.save(filename)
    print(f'🔊 Playing: "{response_text}"')
    display(Audio(filename, autoplay=True))
    return filename

# Test it
speak_response('नमस्ते! आपका प्रश्न सुना गया है। उत्तर खोजा जा रहा है।', 'greeting.mp3')

🔊 Playing: "नमस्ते! आपका प्रश्न सुना गया है। उत्तर खोजा जा रहा है।"


'greeting.mp3'

In [53]:
print('=' * 65)
print('🎙️  HINDI VOICE-TO-VOICE RAG DEMO')
print('=' * 65)

# STAGE 1: ASR
print(f'\n[1/3] 🎙️  INPUT AUDIO : {audio_filename}')
print(f'      📝 TRANSCRIBED : {transcribed_query}')

# STAGE 2: Retrieval
top_results, retrieval_ms = retrieve(transcribed_query, k=2)
best_context, best_dist = top_results[0]

print(f'\n[2/3] 🔍 RETRIEVAL   ({retrieval_ms:.1f}ms)')
for rank, (ctx, dist) in enumerate(top_results, 1):
    print(f'      [{rank}] dist={dist:.3f} | {ctx}')

# STAGE 3: TTS Response
# Build a natural-sounding Hindi response
response_text = f'{best_context}'

print(f'\n[3/3] 🔊 SPOKEN RESPONSE:')
print(f'      "{response_text}"')
print()

speak_response(response_text, 'final_response.mp3')

print('\n' + '='*65)
print('✅ Pipeline complete')
print(f'   ASR (whisper-small) : {t_small:.1f}s')
print(f'   Retrieval (FAISS)   : {retrieval_ms:.1f}ms')
print(f'   TTS (gTTS)          : ~1–2s (network call)')
print('='*65)

🎙️  HINDI VOICE-TO-VOICE RAG DEMO

[1/3] 🎙️  INPUT AUDIO : Audio10 (2).mp3
      📝 TRANSCRIBED : वारनसी में सबसे प्रसिड घात कौनसा है?

[2/3] 🔍 RETRIEVAL   (19.0ms)
      [1] dist=10.527 | वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।
      [2] dist=11.216 | वाराणसी में सबसे प्रसिद्ध घाट दशाश्वमेध घाट है, जहाँ प्रतिदिन गंगा आरती होती है।

[3/3] 🔊 SPOKEN RESPONSE:
      "वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।"

🔊 Playing: "वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।"



✅ Pipeline complete
   ASR (whisper-small) : 2.2s
   Retrieval (FAISS)   : 19.0ms
   TTS (gTTS)          : ~1–2s (network call)


In [54]:
test_queries = [
    'वाराणसी में सबसे प्रसिद्ध घाट कौन सा है?',
    'हनुमान जी ने राम जी की मदद कैसे की?',
    'गंगा नदी कहाँ से निकलती है?',
    'बनारस में कौन सा विश्वविद्यालय है?',
]

print('='*65)
print('🧪 BATCH RETRIEVAL TEST')
print('='*65)
for q in test_queries:
    results, lat = retrieve(q, k=1)
    print(f'Q: {q}')
    print(f'A: {results[0][0]}  ({lat:.1f}ms)')
    print()

🧪 BATCH RETRIEVAL TEST
Q: वाराणसी में सबसे प्रसिद्ध घाट कौन सा है?
A: वाराणसी का सबसे प्रसिद्ध और मुख्य घाट दशाश्वमेध घाट है।  (15.8ms)

Q: हनुमान जी ने राम जी की मदद कैसे की?
A: हनुमान जी भगवान राम के परम भक्त और सेवक थे।  (10.6ms)

Q: गंगा नदी कहाँ से निकलती है?
A: गंगा नदी हिमालय के गंगोत्री ग्लेशियर से निकलती है।  (10.5ms)

Q: बनारस में कौन सा विश्वविद्यालय है?
A: बनारस हिंदू विश्वविद्यालय (BHU) वाराणसी में स्थित है और एशिया के सबसे बड़े विश्वविद्यालयों में से एक है।  (10.6ms)



In [55]:
import platform, torch

# Measure avg retrieval latency
N = 100
q_emb = embedder.encode(['हनुमान जी कौन थे?'], show_progress_bar=False)
q_emb = np.array(q_emb, dtype='float32')
t0 = time.time()
for _ in range(N):
    index.search(q_emb, k=1)
avg_retrieval_ms = (time.time() - t0) / N * 1000

print('\n' + '='*65)
print('📊 PERFORMANCE SUMMARY — paste into README')
print('='*65)
print(f'Hardware              : {"GPU (T4)" if torch.cuda.is_available() else "CPU"}')
print(f'whisper-base  WER     : {wer_base}%  |  latency: {t_base:.1f}s')
print(f'whisper-small WER     : {wer_small}%  |  latency: {t_small:.1f}s')
print(f'FAISS retrieval (avg) : {avg_retrieval_ms:.2f}ms  (over {N} queries)')
print(f'Knowledge base size   : {len(contexts)} sentences')
print(f'Embedding dim         : {dim}')
print('\nNext steps to hit real-time on Raspberry Pi:')
print('  • faster-whisper (CTranslate2 backend) — 4–8× faster than openai-whisper')
print('  • INT8 quantization reduces model size by ~75%')
print('  • whisper.cpp runs directly on Pi 4 ARM CPU')
print('  • Swap gTTS for offline Coqui TTS (no network dependency)')
print('  • Upgrade embedder to LaBSE or multilingual-e5 for native Hindi semantics')
print('='*65)


📊 PERFORMANCE SUMMARY — paste into README
Hardware              : GPU (T4)
whisper-base  WER     : 100.0%  |  latency: 1.3s
whisper-small WER     : 75.0%  |  latency: 2.2s
FAISS retrieval (avg) : 0.01ms  (over 100 queries)
Knowledge base size   : 16 sentences
Embedding dim         : 384

Next steps to hit real-time on Raspberry Pi:
  • faster-whisper (CTranslate2 backend) — 4–8× faster than openai-whisper
  • INT8 quantization reduces model size by ~75%
  • whisper.cpp runs directly on Pi 4 ARM CPU
  • Swap gTTS for offline Coqui TTS (no network dependency)
  • Upgrade embedder to LaBSE or multilingual-e5 for native Hindi semantics
